**MAESTRÍA EN ECONOMÍA APLICADA - UBA 2025**
**TALLER DE PROGRAMACIÓN**
**GRUPO 2**

# PROYECTO FINAL: TALLER DE PROGRAMACIÓN

1) IMPORTS + CONFIGURACIÓN GENERAL

In [7]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances
from scipy.optimize import linear_sum_assignment
import glob


2) FUNCIONES PARA CARGA DE BASES MENSUALES

In [8]:
BASE_PATH = r"C:\Users\marti\Downloads\Bases"

def cargar_bases(base_path=BASE_PATH):
    """
    Busca archivos dentro de carpetas mensuales, carga CSV o Excel.
    """
    dfs = []
    
    for year in range(2019, 2026):
        path_year = os.path.join(base_path, str(year))
        
        if not os.path.exists(path_year):
            continue
        
        # Buscar carpetas StockArgenprop_*
        carpetas = glob.glob(os.path.join(path_year, "StockArgenprop_*"))
        
        for carpeta in carpetas:
            # Si es carpeta, buscar archivos adentro
            if os.path.isdir(carpeta):
                archivos = glob.glob(os.path.join(carpeta, "*"))
            else:
                archivos = [carpeta]
            
            for file in archivos:
                # Filtrar solo formatos soportados
                ext = os.path.splitext(file)[1].lower()
                
                if ext in [".csv"]:
                    try:
                        df = pd.read_csv(file, low_memory=False, encoding="utf-8")
                        df["ArchivoOrigen"] = os.path.basename(file)
                        dfs.append(df)
                    except Exception as e:
                        print("Error cargando CSV:", file, e)

                elif ext in [".xlsx", ".xls"]:
                    try:
                        df = pd.read_excel(file)
                        df["ArchivoOrigen"] = os.path.basename(file)
                        dfs.append(df)
                    except Exception as e:
                        print("Error cargando Excel:", file, e)

                else:
                    # No es archivo cargable
                    continue

    if len(dfs) == 0:
        raise ValueError("No se encontraron archivos CSV ni Excel en las carpetas.")
    
    df_total = pd.concat(dfs, ignore_index=True)
    print(f"✔ Bases cargadas: {len(dfs)} archivos")
    print(f"✔ Total de registros: {df_total.shape[0]}")
    
    return df_total


3) LIMPIEZA + GENERACIÓN DE VARIABLES

Incluye:

manejo de precios, superficies

precio por m2

coordenadas

antigüedad

ambientes

manejo básico de missing

generación de month

In [9]:
def preparar_dataframe(df):
    # Convertir fechas
    df["FechaPublicacion"] = pd.to_datetime(df["FechaPublicacion"], errors='coerce')
    df["FechaModificacion"] = pd.to_datetime(df["FechaModificacion"], errors='coerce')

    # Definir mes (último día hábil, pero acá tomamos mes calendario)
    df["month"] = df["FechaModificacion"].dt.to_period("M")

    # Limpiar superficies
    for col in [
        "PropiedadSuperficieTotal",
        "SuperficieCubierta",
        "SuperficieDescubierta"
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Convertir precio a numérico
    df["MontoOperacion"] = pd.to_numeric(df["MontoOperacion"], errors="coerce")

    # Calcular precio m2
    df["precio_m2"] = df["MontoOperacion"] / df["PropiedadSuperficieTotal"].replace(0, np.nan)
    df["precio_m2"] = df["precio_m2"].fillna(df["precio_m2"].median())

    # Variables numéricas básicas
    df["CantidadAmbientes"] = pd.to_numeric(df["CantidadAmbientes"], errors="coerce")
    df["Antiguedad"] = pd.to_numeric(df["Antiguedad"], errors="coerce")
    df["CantidadDormitorios"] = pd.to_numeric(df["CantidadDormitorios"], errors="coerce")
    df["DepartamentoExpensas"] = pd.to_numeric(df["DepartamentoExpensas"], errors="coerce")

    # Coordenadas
    df["Latitud"] = pd.to_numeric(df["Latitud"], errors="coerce")
    df["Longitud"] = pd.to_numeric(df["Longitud"], errors="coerce")

    # Reemplazar missing básicos
    df.fillna({
        "CantidadAmbientes": df["CantidadAmbientes"].median(),
        "Antiguedad": df["Antiguedad"].median(),
        "CantidadDormitorios": df["CantidadDormitorios"].median(),
        "DepartamentoExpensas": df["DepartamentoExpensas"].median(),
    }, inplace=True)

    return df


4) ARMADO DE FEATURE SET PARA EL CLUSTERING

In [10]:
def construir_features(df):
    """
    Devuelve X (matriz numérica) y df con solo las columnas útiles.
    """
    features = [
        "precio_m2",
        "PropiedadSuperficieTotal",
        "SuperficieCubierta",
        "CantidadAmbientes",
        "Antiguedad",
        "Latitud",
        "Longitud"
    ]

    X = df[features].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, features


5) CLUSTERING MENSUAL + MATCHING ENTRE MESES

In [11]:
def clustering_dinamico(df, k=6):
    meses = sorted(df["month"].unique())
    
    centroids_by_month = {}
    labels_by_month = {}
    
    for m in meses:
        dfm = df[df["month"] == m]
        if len(dfm) < k:
            continue
        
        X, _ = construir_features(dfm)

        km = MiniBatchKMeans(n_clusters=k, random_state=0, batch_size=500)
        labels = km.fit_predict(X)
        centroids = km.cluster_centers_
        
        centroids_by_month[m] = centroids
        labels_by_month[m] = pd.Series(labels, index=dfm.index)
    
    # MATCHING ENTRE MESES
    cluster_id_global = {}
    current_global_ids = list(range(k))  # primeros IDs
    
    for i in range(len(meses) - 1):
        m = meses[i]
        m_next = meses[i + 1]

        if m not in centroids_by_month or m_next not in centroids_by_month:
            continue
        
        C0 = centroids_by_month[m]
        C1 = centroids_by_month[m_next]

        D = pairwise_distances(C0, C1)
        row_ind, col_ind = linear_sum_assignment(D)

        # asignar IDs globales
        mapping = {int(col_ind[j]): current_global_ids[j] for j in range(len(col_ind))}
        cluster_id_global[m_next] = mapping

    # Aplicar IDs globales
    df["cluster_id"] = np.nan
    for m in meses:
        if m in labels_by_month:
            df.loc[labels_by_month[m].index, "cluster_id"] = labels_by_month[m]

    # Reemplazar por IDs globales
    for m in meses[1:]:
        if m not in cluster_id_global:
            continue
        mapping = cluster_id_global[m]
        df.loc[df["month"] == m, "cluster_id"] = \
            df.loc[df["month"] == m, "cluster_id"].astype(int).map(mapping)

    return df


6) PIPELINE COMPLETO (lo corrés una sola vez)

In [12]:
# 1) Cargar
df = cargar_bases()

# 2) Limpiar
df = preparar_dataframe(df)

# 3) Aplicar clustering dinámico
df_clusterizado = clustering_dinamico(df, k=6)

# 4) Guardar resultados
df_clusterizado.to_csv("resultados_clusters_dinamicos.csv", index=False)

df_clusterizado.head()


KeyboardInterrupt: 